# Connoisseur Companion v1: Qwen con Hugging Face

Bucle ReAct manual con Qwen2.5-7B-Instruct mediante Hugging Face. El token gratuito se lee desde HF_TOKEN. Coloca server.py y los datos MCP junto a este notebook.

## Instalación

In [ ]:
# %pip install -q gradio fastmcp huggingface_hub
# Instalacion mediante el archivo requirements.txt

## Configuración

In [1]:
import os, json
from pathlib import Path
from typing import Any, cast
from huggingface_hub import (
  AsyncInferenceClient,
  ChatCompletionInputFunctionDefinition,
  ChatCompletionInputTool,
)
#from fastmcp import Client, PythonStdioTransport
from fastmcp import Client
from fastmcp.client.transports import PythonStdioTransport
SERVER_SCRIPT = str(Path.cwd() / 'server.py')
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN: raise RuntimeError('Define HF_TOKEN antes de continuar.')
llm = AsyncInferenceClient(token=HF_TOKEN)
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
SYSTEM_PROMPT = '''Eres Connoisseur Companion y respondes en castellano. Consulta MCP para restaurantes de California. Usa get_restaurant_info por nombre, recommend_by_vibe por ambiente y get_review para reseñas. No inventes datos.'''


## Agente ReAct y MCP

In [2]:
def texto(resultado): return '\n'.join(getattr(i, 'text', str(i)) for i in (resultado.content or [])) or '(sin resultado)'
MCP_LOG = Path.cwd() / 'mcp-server.log'
async def chat_with_agent(mensaje, historial=None):
  transporte = PythonStdioTransport(script_path=SERVER_SCRIPT, log_file=MCP_LOG)
  async with Client(transporte) as cliente:
    descubiertas = await cliente.list_tools()
    tools: list[ChatCompletionInputTool] = [
      cast(ChatCompletionInputTool, {
        'type': 'function',
        'function': cast(ChatCompletionInputFunctionDefinition, {
          'name': t.name,
          'description': t.description or '',
          'parameters': t.input_schema,
        }),
      })
      for t in descubiertas
    ]
    mensajes: list[dict[str, Any]] = [{'role':'system','content':SYSTEM_PROMPT}] + list(historial or []) + [{'role':'user','content':mensaje}]
    for _ in range(8):
      respuesta_api = await llm.chat_completion(model=MODEL,messages=mensajes,tools=tools,tool_choice='auto',temperature=0.2,stream=False)
      respuesta = respuesta_api.choices[0].message
      tool_calls = getattr(respuesta, 'tool_calls', None) or []
      mensaje_asistente: dict[str, Any] = {'role':'assistant','content':respuesta.content or ''}
      if tool_calls:
        mensaje_asistente['tool_calls'] = [{'id':llamada.id,'type':'function','function':{'name':llamada.function.name,'arguments':llamada.function.arguments}} for llamada in tool_calls]
      mensajes.append(mensaje_asistente)
      if not tool_calls: return respuesta.content or '(respuesta vacía)'
      for llamada in tool_calls:
        datos = json.loads(llamada.function.arguments or '{}')
        resultado = await cliente.call_tool(llamada.function.name, datos)
        mensajes.append({'role':'tool','tool_call_id':llamada.id,'content':texto(resultado)})
    return 'No pude completar la consulta.'

In [3]:
async def diagnosticar_mcp():
  transporte = PythonStdioTransport(script_path=SERVER_SCRIPT, log_file=MCP_LOG)
  async with Client(transporte) as cliente:
    herramientas = await cliente.list_tools()
    print('Herramientas:', [herramienta.name for herramienta in herramientas])
    resultado = await cliente.call_tool('recommend_by_vibe', {'vibe': 'romantico'})
    print('Resultado:', texto(resultado))

await diagnosticar_mcp()

Herramientas: ['get_restaurant_info', 'recommend_by_vibe', 'get_review']
Resultado: {
  "vibe_searched": "romantico",
  "structured_matches": [],
  "raw_text_excerpts": []
}


## Interfaz Gradio

In [ ]:
import gradio as gr
async def manejar(mensaje, historial):
  historial = historial or []
  return historial + [{'role':'user','content':mensaje},{'role':'assistant','content':await chat_with_agent(mensaje,historial)}]
with gr.Blocks(title='Connoisseur Companion v1') as demo:
  gr.Markdown('# Connoisseur Companion v1\nQwen + MCP')
  chat=gr.Chatbot(height=500)
  entrada=gr.Textbox(label='Consulta')
  entrada.submit(manejar,[entrada,chat],chat).then(lambda:'',None,entrada)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


r:\Home\juanr\repo\IA-Agentica-Multiagente\.venv\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
r:\Home\juanr\repo\IA-Agentica-Multiagente\.venv\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
